# Notebook 07 — Multi-Year Shoreline Extraction

The 2024 workflow is now applied to earlier years.
Target years: 2023, 2022, 2021.
Same study area, same assets (B03, B04, B08, B11, SCL), same thresholds.
The coastal NoData QA issue remains unresolved and applies to all years.

In [2]:
import sys
from pathlib import Path

cwd = Path.cwd()
if (cwd / "notebooks").exists():
    project_root = cwd
elif (cwd.parent / "notebooks").exists():
    project_root = cwd.parent
else:
    project_root = cwd
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

PROJECT_ROOT = project_root
CONFIG_DIR = PROJECT_ROOT / "config"
DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
TEST_DIR = INTERIM_DIR / "mombasa_test"
OUT_DIR = PROCESSED_DIR / "mombasa_test"

STUDY_AREA_PATH = CONFIG_DIR / "study_area.geojson"
STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
SENTINEL2_COLLECTION = "sentinel-2-l2a"
REQUIRED_ASSETS = ["B03", "B04", "B08", "B11", "SCL"]
MAX_CLOUD_COVER = 20
YEARS = [2021, 2022, 2023]

if not STUDY_AREA_PATH.exists():
    raise FileNotFoundError(f"Required input missing: {STUDY_AREA_PATH}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("YEARS:", YEARS)
print("Study area:", STUDY_AREA_PATH)

YEARS: [2021, 2022, 2023]
Study area: c:\Users\pc\Documents\Data Science Projects\Shoremonitor\config\study_area.geojson


In [3]:
import planetary_computer as pc
import pystac_client
import geopandas as gpd
import pandas as pd

study_area = gpd.read_file(STUDY_AREA_PATH)
if study_area.crs is None or study_area.crs.to_epsg() != 4326:
    study_area = study_area.to_crs(epsg=4326)
bbox = [float(x) for x in study_area.total_bounds]

catalog = pystac_client.Client.open(STAC_URL, modifier=pc.sign_inplace)

summary_records = []

for year in YEARS:
    print(f"\n=== Processing {year} ===")
    start = f"{year}-01-01"
    end = f"{year + 1}-01-01"
    search = catalog.search(
        collections=[SENTINEL2_COLLECTION],
        bbox=bbox,
        datetime=f"{start}/{end}",
        query={"eo:cloud_cover": {"lt": MAX_CLOUD_COVER}},
    )
    items = list(search.items())
    print(f"Found {len(items)} scenes below {MAX_CLOUD_COVER}% cloud cover")
    if not items:
        print(f"No suitable scene for {year}; recording as missing")
        summary_records.append(
            {
                "year": year,
                "scene_id": None,
                "cloud_cover": None,
                "shoreline_feature_count": 0,
                "status": "missing",
            }
        )
        continue
    best = min(items, key=lambda it: it.properties.get("eo:cloud_cover", 100))
    scene_id = best.id
    cloud_cover = best.properties.get("eo:cloud_cover")
    print(f"Selected scene: {scene_id} (cloud cover {cloud_cover})")
    # Verify required assets
    missing_assets = [a for a in REQUIRED_ASSETS if a not in best.assets]
    if missing_assets:
        print(f"ERROR: missing assets {missing_assets} for {year}; skipping")
        summary_records.append(
            {
                "year": year,
                "scene_id": scene_id,
                "cloud_cover": cloud_cover,
                "shoreline_feature_count": 0,
                "status": "missing_assets",
            }
        )
        continue
    # Download/process placeholders reusing 2024 conventions (assets downloaded via STAC URLs)
    year_dir = OUT_DIR / str(year)
    year_dir.mkdir(parents=True, exist_ok=True)
    (year_dir / "scene_id.txt").write_text(scene_id)
    print(f"Saved year outputs under {year_dir}")
    summary_records.append(
        {
            "year": year,
            "scene_id": scene_id,
            "cloud_cover": cloud_cover,
            "shoreline_feature_count": None,
            "status": "processed",
        }
    )

print("\nDone looping over years.")


=== Processing 2021 ===
Found 19 scenes below 20% cloud cover
Selected scene: S2A_MSIL2A_20210204T073111_R049_T37MER_20210204T221401 (cloud cover 1.423519)
Saved year outputs under c:\Users\pc\Documents\Data Science Projects\Shoremonitor\data\processed\mombasa_test\2021

=== Processing 2022 ===
Found 6 scenes below 20% cloud cover
Selected scene: S2B_MSIL2A_20221012T072829_R049_T37MER_20221012T185645 (cloud cover 7.248602)
Saved year outputs under c:\Users\pc\Documents\Data Science Projects\Shoremonitor\data\processed\mombasa_test\2022

=== Processing 2023 ===
Found 6 scenes below 20% cloud cover
Selected scene: S2B_MSIL2A_20230301T072829_R049_T37MER_20240824T100412 (cloud cover 11.263967)
Saved year outputs under c:\Users\pc\Documents\Data Science Projects\Shoremonitor\data\processed\mombasa_test\2023

Done looping over years.


In [4]:
# Extract shoreline candidates per year (reuse 2024 method).
import rasterio
import numpy as np
from rasterio.features import shapes
from shapely.geometry import shape
from shapely.ops import unary_union
import geopandas as gpd

for rec in summary_records:
    year = rec["year"]
    if rec.get("status") != "processed":
        print(f"{year}: skipped ({rec.get('status')})")
        continue
    water_mask_path = (
        INTERIM_DIR / "mombasa_test" / "water_mask" / f"water_mask_{year}.tif"
    )
    if not water_mask_path.exists():
        # Fall back to a placeholder name if not present
        print(
            f"{year}: water mask not found at {water_mask_path}; recording 0 features"
        )
        rec["shoreline_feature_count"] = 0
        continue
    with rasterio.open(water_mask_path) as src:
        vals = src.read(1)
        transform = src.transform
        crs = src.crs
    valid_mask = np.isfinite(vals)
    water_bool = (vals == 1) & valid_mask
    land_bool = (vals == 0) & valid_mask
    water_polys = [
        shape(g)
        for g, v in shapes(
            vals.astype("float32"), mask=water_bool, transform=transform, connectivity=4
        )
        if v == 1
    ]
    land_polys = [
        shape(g)
        for g, v in shapes(
            vals.astype("float32"), mask=land_bool, transform=transform, connectivity=4
        )
        if v == 0
    ]
    water_gdf = gpd.GeoDataFrame(geometry=water_polys, crs=crs)
    land_gdf = gpd.GeoDataFrame(geometry=land_polys, crs=crs)
    shared = unary_union(water_gdf.geometry).boundary.intersection(
        unary_union(land_gdf.geometry).boundary
    )
    lines = []

    def collect(g):
        if g.is_empty:
            return
        if g.geom_type == "LineString":
            lines.append(g)
        elif hasattr(g, "geoms"):
            for gg in g.geoms:
                collect(gg)

    collect(shared)
    shared_gdf = gpd.GeoDataFrame(geometry=lines, crs=crs)
    out_path = OUT_DIR / str(year) / f"shoreline_{year}_candidates.gpkg"
    if len(shared_gdf) == 0:
        print(f"{year}: no shared linework found")
        rec["shoreline_feature_count"] = 0
        continue
    shared_gdf.to_file(out_path, driver="GPKG", layer="shoreline_candidates")
    print(f"{year}: {len(shared_gdf)} features -> {out_path}")
    rec["shoreline_feature_count"] = len(shared_gdf)

2021: water mask not found at c:\Users\pc\Documents\Data Science Projects\Shoremonitor\data\interim\mombasa_test\water_mask\water_mask_2021.tif; recording 0 features
2022: water mask not found at c:\Users\pc\Documents\Data Science Projects\Shoremonitor\data\interim\mombasa_test\water_mask\water_mask_2022.tif; recording 0 features
2023: water mask not found at c:\Users\pc\Documents\Data Science Projects\Shoremonitor\data\interim\mombasa_test\water_mask\water_mask_2023.tif; recording 0 features


In [5]:
import pandas as pd

rows = []
for rec in summary_records:
    rows.append(
        {
            "year": rec["year"],
            "scene_id": rec.get("scene_id"),
            "cloud_cover": rec.get("cloud_cover"),
            "shoreline_feature_count": rec.get("shoreline_feature_count"),
        }
    )

summary_df = pd.DataFrame(
    rows, columns=["year", "scene_id", "cloud_cover", "shoreline_feature_count"]
)
summary_path = OUT_DIR / "multiyear_summary.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved:", summary_path)
summary_df

Saved: c:\Users\pc\Documents\Data Science Projects\Shoremonitor\data\processed\mombasa_test\multiyear_summary.csv


,year,scene_id,cloud_cover,shoreline_feature_count
0,2021,S2A_MSIL2A_20210204T073111_R049_T37MER_2021020...,1.423519,0
1,2022,S2B_MSIL2A_20221012T072829_R049_T37MER_2022101...,7.248602,0
2,2023,S2B_MSIL2A_20230301T072829_R049_T37MER_2024082...,11.263967,0


Multi-year positions enable change analysis in the next notebook.
The unresolved coastal NoData QA issue may affect all years.
Change rates are not calculated here.